# Flyte Hello World - Introduccion a ML Pipelines

Este notebook demuestra como usar Flyte para orquestar workflows de Machine Learning.

## Contenido
1. Configuracion del entorno
2. Definicion de tareas (tasks)
3. Definicion de workflows
4. Ejecucion local
5. Ejecucion remota en el cluster

## 1. Configuracion del entorno

In [ ]:
# Instalar flytekit si no esta instalado
# !pip install flytekit
# o con uv:
# !uv add flytekit

In [ ]:
import flytekit
from flytekit import task, workflow, dynamic
from flytekit.types.file import FlyteFile
from typing import List, Tuple
import pandas as pd

print(f"Flytekit version: {flytekit.__version__}")

## 2. Definicion de Tareas (Tasks)

Las tareas son las unidades basicas de trabajo en Flyte. Cada tarea es una funcion de Python decorada con `@task`.

In [ ]:
@task
def say_hello(name: str) -> str:
    """
    Tarea simple que saluda a una persona.
    
    Args:
        name: Nombre de la persona a saludar
    
    Returns:
        Mensaje de saludo
    """
    greeting = f"Hello, {name}! Welcome to Flyte."
    print(greeting)
    return greeting

In [ ]:
@task
def add_numbers(a: int, b: int) -> int:
    """
    Suma dos numeros.
    
    Args:
        a: Primer numero
        b: Segundo numero
    
    Returns:
        Suma de a y b
    """
    result = a + b
    print(f"{a} + {b} = {result}")
    return result

In [ ]:
@task
def multiply_by_factor(number: int, factor: int = 2) -> int:
    """
    Multiplica un numero por un factor.
    
    Args:
        number: Numero a multiplicar
        factor: Factor de multiplicacion
    
    Returns:
        Resultado de la multiplicacion
    """
    result = number * factor
    print(f"{number} * {factor} = {result}")
    return result

## 3. Definicion de Workflows

Los workflows combinan multiples tareas en un pipeline. Se definen con el decorador `@workflow`.

In [ ]:
@workflow
def hello_workflow(name: str) -> str:
    """
    Workflow simple que saluda a una persona.
    
    Args:
        name: Nombre de la persona
    
    Returns:
        Mensaje de saludo
    """
    return say_hello(name=name)

In [ ]:
@workflow
def math_workflow(a: int, b: int, factor: int = 3) -> int:
    """
    Workflow que suma dos numeros y multiplica el resultado.
    
    Pipeline: (a + b) * factor
    
    Args:
        a: Primer numero
        b: Segundo numero
        factor: Factor de multiplicacion
    
    Returns:
        Resultado final
    """
    # Primero sumamos
    sum_result = add_numbers(a=a, b=b)
    
    # Luego multiplicamos
    final_result = multiply_by_factor(number=sum_result, factor=factor)
    
    return final_result

## 4. Ejecucion Local

Antes de ejecutar en el cluster, podemos probar los workflows localmente.

In [ ]:
# Probar la tarea de saludo
result = say_hello(name="DSRP")
print(f"Resultado: {result}")

In [ ]:
# Probar el workflow de saludo
result = hello_workflow(name="ML Engineer")
print(f"\nResultado del workflow: {result}")

In [ ]:
# Probar el workflow matematico
# (5 + 3) * 4 = 32
result = math_workflow(a=5, b=3, factor=4)
print(f"\nResultado final: {result}")

## 5. Ejemplo de ML Pipeline

Un ejemplo mas realista de un pipeline de Machine Learning.

In [ ]:
import numpy as np
from dataclasses import dataclass
from dataclasses_json import dataclass_json


@dataclass_json
@dataclass
class ModelMetrics:
    """Metricas del modelo."""
    accuracy: float
    precision: float
    recall: float
    f1_score: float

In [ ]:
@task
def generate_data(n_samples: int = 100, n_features: int = 5) -> Tuple[np.ndarray, np.ndarray]:
    """
    Genera datos sinteticos para clasificacion.
    
    Args:
        n_samples: Numero de muestras
        n_features: Numero de features
    
    Returns:
        Tupla con features (X) y etiquetas (y)
    """
    np.random.seed(42)
    
    # Generar features aleatorias
    X = np.random.randn(n_samples, n_features)
    
    # Generar etiquetas binarias basadas en una regla simple
    y = (X[:, 0] + X[:, 1] > 0).astype(int)
    
    print(f"Datos generados: {n_samples} muestras, {n_features} features")
    print(f"Distribucion de clases: {np.bincount(y)}")
    
    return X, y

In [ ]:
@task
def train_model(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Entrena un modelo simple (regresion logistica simulada).
    
    Args:
        X: Features de entrenamiento
        y: Etiquetas de entrenamiento
    
    Returns:
        Pesos del modelo
    """
    # Simulacion simple: pesos aleatorios ajustados
    n_features = X.shape[1]
    weights = np.random.randn(n_features) * 0.1
    
    # "Entrenamiento" simple: ajustar hacia la correlacion real
    weights[0] = 0.5  # Peso para feature 0
    weights[1] = 0.5  # Peso para feature 1
    
    print(f"Modelo entrenado con {n_features} pesos")
    print(f"Pesos: {weights}")
    
    return weights

In [ ]:
@task
def evaluate_model(X: np.ndarray, y: np.ndarray, weights: np.ndarray) -> ModelMetrics:
    """
    Evalua el modelo y calcula metricas.
    
    Args:
        X: Features de evaluacion
        y: Etiquetas reales
        weights: Pesos del modelo
    
    Returns:
        Metricas del modelo
    """
    # Predicciones
    scores = X @ weights
    predictions = (scores > 0).astype(int)
    
    # Calcular metricas
    tp = np.sum((predictions == 1) & (y == 1))
    tn = np.sum((predictions == 0) & (y == 0))
    fp = np.sum((predictions == 1) & (y == 0))
    fn = np.sum((predictions == 0) & (y == 1))
    
    accuracy = (tp + tn) / len(y)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    metrics = ModelMetrics(
        accuracy=float(accuracy),
        precision=float(precision),
        recall=float(recall),
        f1_score=float(f1)
    )
    
    print(f"Metricas del modelo:")
    print(f"  - Accuracy: {metrics.accuracy:.4f}")
    print(f"  - Precision: {metrics.precision:.4f}")
    print(f"  - Recall: {metrics.recall:.4f}")
    print(f"  - F1-Score: {metrics.f1_score:.4f}")
    
    return metrics

In [ ]:
@workflow
def ml_training_pipeline(n_samples: int = 200, n_features: int = 5) -> ModelMetrics:
    """
    Pipeline completo de entrenamiento de ML.
    
    Pasos:
    1. Generar datos
    2. Entrenar modelo
    3. Evaluar modelo
    
    Args:
        n_samples: Numero de muestras a generar
        n_features: Numero de features
    
    Returns:
        Metricas del modelo entrenado
    """
    # Paso 1: Generar datos
    X, y = generate_data(n_samples=n_samples, n_features=n_features)
    
    # Paso 2: Entrenar modelo
    weights = train_model(X=X, y=y)
    
    # Paso 3: Evaluar modelo
    metrics = evaluate_model(X=X, y=y, weights=weights)
    
    return metrics

In [ ]:
# Ejecutar el pipeline de ML localmente
metrics = ml_training_pipeline(n_samples=500, n_features=10)
print(f"\nResultado final: {metrics}")

## 6. Ejecucion Remota en el Cluster

Para ejecutar workflows en el cluster de Flyte desplegado en AKS.

In [ ]:
# Configuracion para ejecucion remota
# Descomenta y ajusta segun tu configuracion

# from flytekit.remote import FlyteRemote
# from flytekit.configuration import Config

# # Configurar conexion al cluster
# FLYTE_ENDPOINT = "<IP_PUBLICA>:8089"  # Reemplazar con tu IP

# remote = FlyteRemote(
#     config=Config.for_endpoint(
#         endpoint=FLYTE_ENDPOINT,
#         insecure=True  # Para sandbox sin SSL
#     ),
#     default_project="flytesnacks",
#     default_domain="development",
# )

In [ ]:
# Registrar y ejecutar workflow en el cluster
# Descomenta para usar

# # Registrar el workflow
# registered_wf = remote.register_workflow(ml_training_pipeline)
# print(f"Workflow registrado: {registered_wf.id}")

# # Ejecutar el workflow
# execution = remote.execute(
#     registered_wf,
#     inputs={"n_samples": 1000, "n_features": 10},
#     wait=True  # Esperar a que termine
# )

# # Obtener resultados
# print(f"Estado: {execution.closure.phase}")
# print(f"Resultados: {execution.outputs}")

## 7. Alternativa: Usar pyflyte desde terminal

Tambien puedes registrar y ejecutar workflows desde la terminal:

```bash
# Registrar el workflow
pyflyte register flyte_hello_world.py --project flytesnacks --domain development

# Ejecutar el workflow
pyflyte run --remote flyte_hello_world.py ml_training_pipeline --n_samples 500
```

## Resumen

En este notebook aprendimos:

1. **Tasks**: Funciones individuales decoradas con `@task`
2. **Workflows**: Composicion de tasks decorada con `@workflow`
3. **Ejecucion local**: Probar workflows antes de desplegar
4. **Tipos de datos**: Usar tipos nativos de Python y dataclasses
5. **Ejecucion remota**: Conectar a un cluster de Flyte

### Proximos pasos

- Explorar mas features de Flyte: caching, retries, recursos
- Integrar con MLflow para tracking de experimentos
- Crear pipelines mas complejos con branching y loops